# T2-C1: synthetic sensitivity calibration

Reproduce the fixed experiment from 12 September 2026. Upload **T2_C1_Sensitivity.zip** when prompted. This notebook checks the archive, verifies the probe calculation, and reruns all 24 panels into a separate output folder. No GPU, Transformer checkpoint, or reserved model seed is used.

The archived result is 0/24 full-screen passes at known population gains 0 and 0.01, and 24/24 at each of 0.05 and 0.10. This calibrates simple synthetic signals; it does not establish a natural Transformer history advantage. Read `T2_C1_Report.md` for uncertainty and limitations.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import hashlib, json, subprocess, sys, tempfile, zipfile

uploaded = files.upload()
assert set(uploaded) == {'T2_C1_Sensitivity.zip'}, 'Upload exactly the provided ZIP bundle.'
archive = Path('T2_C1_Sensitivity.zip')
work = Path(tempfile.mkdtemp(prefix='t2_c1_'))
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        name = PurePosixPath(member.filename)
        assert not name.is_absolute() and '..' not in name.parts
        assert (member.external_attr >> 16) & 0o170000 != 0o120000, 'Symlinks are not accepted.'
        assert (work / member.filename).resolve().is_relative_to(work.resolve())
    z.extractall(work)
root = work / 'T2_C1_Sensitivity'
manifest = json.loads((root / 'PACKAGE_SHA256.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256((root / name).read_bytes()).hexdigest() == expected, name
print(f'Verified {len(manifest)} files in {root}')


## Install and verify

The fixed scientific dependencies are installed in the runtime. All experiment commands run in fresh Python subprocesses, so previously imported notebook packages do not affect the runner.


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(root / 'requirements.txt')], check=True)
subprocess.run([sys.executable, 'verify_methods.py'], cwd=root, check=True)
subprocess.run([sys.executable, 'verify_results.py'], cwd=root, check=True)


## Reproduce the complete fixed experiment

This regenerates 144 datasets and evaluates four paired target scenarios per dataset. It is a reproduction of the archived run, not a new independent panel sample. Runtime varies by CPU; the original run took about 44 seconds.


In [ ]:
reproduced = root / 'reproduced_results'
subprocess.run([sys.executable, '-u', 'run_calibration.py', '--out', str(reproduced)], cwd=root, check=True)
subprocess.run([sys.executable, 'verify_results.py', '--results', str(reproduced)], cwd=root, check=True)


In [ ]:
comparison = r"""
import json
from pathlib import Path
import numpy as np
import pandas as pd
root = Path('.')
old = pd.read_csv(root/'results/dataset_metrics.csv')
new = pd.read_csv(root/'reproduced_results/dataset_metrics.csv')
assert list(old.columns) == list(new.columns)
np.testing.assert_allclose(old.to_numpy(), new.to_numpy(), rtol=1e-7, atol=1e-8)
a = pd.read_csv(root/'results/panel_metrics.csv')
b = pd.read_csv(root/'reproduced_results/panel_metrics.csv')
np.testing.assert_array_equal(a.screen_passed, b.screen_passed)
print('All dataset scores match within tolerance; all 96 panel decisions match.')
summary = json.loads((root/'reproduced_results/detection_summary.json').read_text())
print(pd.DataFrame(summary)[['planted_population_delta_r2', 'detected_panels', 'panels', 'monte_carlo_wilson_ci95']].to_string(index=False))
"""
subprocess.run([sys.executable, '-c', comparison], cwd=root, check=True)


## Inspect and download

The Wilson intervals summarize Monte Carlo uncertainty over 24 panels. Conditional family-bootstrap intervals are stored separately for each panel. A null pass count of zero does not guarantee a false-positive rate below 5%, and this calibration cannot be transferred directly to actual neural representations.


In [ ]:
from IPython.display import Image, Markdown, display
import shutil
display(Image(filename=str(root / 'T2_C1_Sensitivity.png')))
display(Markdown((root / 'T2_C1_Report.md').read_text()))
result_zip = shutil.make_archive(str(work / 'T2_C1_reproduced_results'), 'zip', reproduced)
files.download(result_zip)
